In [1]:
!pip install torch torchvision wandb thop


In [2]:
import wandb
wandb.login()


/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Find your API key here: https://wandb.ai/authorize?ref=models
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: akankshakapil8 (akankshakapil8-iit-jodhpur) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [3]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader


In [4]:
device = "cuda" if torch.cuda.is_available() else "cpu"
device


'cuda'

In [13]:
def get_dataloaders(batch_size=128):

    transform_train = transforms.Compose([
        transforms.RandomHorizontalFlip(),
        transforms.RandomCrop(32, padding=4),
        transforms.ToTensor(),
        transforms.Normalize((0.5, 0.5, 0.5),
                             (0.5, 0.5, 0.5))
    ])

    transform_test = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.5, 0.5, 0.5),
                             (0.5, 0.5, 0.5))
    ])

    trainset = datasets.CIFAR10(
        root="./data",
        train=True,
        download=True,
        transform=transform_train
    )

    testset = datasets.CIFAR10(
        root="./data",
        train=False,
        download=True,
        transform=transform_test
    )

    trainloader = DataLoader(
        trainset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=2
    )

    testloader = DataLoader(
        testset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=2
    )

    return trainloader, testloader


In [14]:
class CIFAR10_CNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.conv1 = nn.Conv2d(3, 32, 3, padding=1)
        self.bn1 = nn.BatchNorm2d(32)

        self.conv2 = nn.Conv2d(32, 32, 3, padding=1)
        self.bn2 = nn.BatchNorm2d(32)

        self.conv3 = nn.Conv2d(32, 64, 3, padding=1)
        self.bn3 = nn.BatchNorm2d(64)

        self.conv4 = nn.Conv2d(64, 64, 3, padding=1)
        self.bn4 = nn.BatchNorm2d(64)

        self.fc1 = nn.Linear(64 * 8 * 8, 256)
        self.fc2 = nn.Linear(256, 10)

    def forward(self, x):
        x = torch.relu(self.bn1(self.conv1(x)))
        x = torch.relu(self.bn2(self.conv2(x)))
        x = torch.max_pool2d(x, 2)

        x = torch.relu(self.bn3(self.conv3(x)))
        x = torch.relu(self.bn4(self.conv4(x)))
        x = torch.max_pool2d(x, 2)

        x = x.view(x.size(0), -1)
        x = torch.relu(self.fc1(x))
        return self.fc2(x)


In [15]:
def evaluate(model, dataloader, device):
    model.eval()
    correct = 0
    total = 0
    running_loss = 0.0
    criterion = nn.CrossEntropyLoss()

    with torch.no_grad():
        for images, labels in dataloader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)

            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
            running_loss += loss.item()

    accuracy = 100 * correct / total
    avg_loss = running_loss / len(dataloader)

    return accuracy, avg_loss


In [20]:
from thop import profile

model = CIFAR10_CNN()
dummy_input = torch.randn(1, 3, 32, 32)

macs, params = profile(model, inputs=(dummy_input,))
print(f"FLOPs: {2 * macs / 1e6:.2f} MFLOPs")
print(f"Parameters: {params / 1e6:.2f} Million")


[INFO] Register count_convNd() for <class 'torch.nn.modules.conv.Conv2d'>.
[INFO] Register count_normalization() for <class 'torch.nn.modules.batchnorm.BatchNorm2d'>.
[INFO] Register count_linear() for <class 'torch.nn.modules.linear.Linear'>.
FLOPs: 51.84 MFLOPs
Parameters: 1.12 Million


In [21]:
def log_gradient_flow(model, epoch):
    for name, param in model.named_parameters():
        if param.grad is not None:
            wandb.log({
                f"Gradients/{name}": param.grad.detach().abs().mean().item(),
                "epoch": epoch
            })

def log_weight_flow(model, epoch):
    for name, param in model.named_parameters():
        wandb.log({
            f"Weights/{name}": param.detach().abs().mean().item(),
            "epoch": epoch
        })


In [24]:
wandb.init(
    project="CIFAR10-CNN-Lab3",
    config={
        "epochs": 30,
        "batch_size": 128,
        "optimizer": "Adam",
        "learning_rate": 1e-3
    }
)


Gradients/bn1.bias,▂▅▃▄▆▁▃▄▅▃▆█▃▃▄▄▅▆▆▅▂▇▂▆▃▅▄
Gradients/bn1.weight,▆▂▂▅▆▄▂▆▅▃▆▅▅▂▃▃▆▇▅█▁█▁▄▆▇▇
Gradients/bn2.bias,▁▁▁▁▂▃▂▃▃▃▄▄▂▂▄▅▂▃▄▄▂█▃▅▂▄▄
Gradients/bn2.weight,▂▁▁▁▂▂▂▆▃▃▅▃▂▃▄▅▃▄▃▄▃█▃▅▄▃▄
Gradients/bn3.bias,▁▂▂▂▃▃▃▄▅▅▄▄▅▄▄▇▄▆▄▆▄█▄▅▅▆▅
Gradients/bn3.weight,▁▂▂▂▃▃▃▄▄▄▄▄▅▄▄▆▅▆▄▆▄█▄▅▄▅▄
Gradients/bn4.bias,▂▁▂▂▁▃▃▅▄▅▅▄▄▆▄▇▃▂▄▇▄█▃▇▅▅▄
Gradients/bn4.weight,▂▁▂▂▂▃▃▅▄▅▅▄▄▆▄▆▄▃▄▇▄█▃▇▆▆▄
Gradients/conv1.bias,▂▃▂▃▄▂▃▅▃▄▄▃▄▂▃▄▂▃▃▅▂█▁▅▃▃▃
Gradients/conv1.weight,▂▇▄▆█▅▃▄▅▅▃▆▇▅▂▅▃▅▅▂▁█▁▁▂▃▂
+35,...


In [25]:
trainloader, testloader = get_dataloaders()

model = CIFAR10_CNN().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

epochs = 30

for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in trainloader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    train_acc = 100 * correct / total
    train_loss = running_loss / len(trainloader)

    # Evaluate on test set
    test_acc, test_loss = evaluate(model, testloader, device)

    # Log gradient & weight flow (once per epoch)
    log_gradient_flow(model, epoch)
    log_weight_flow(model, epoch)

    # Log everything to WandB
    wandb.log({
        "epoch": epoch,
        "train_loss": train_loss,
        "train_accuracy": train_acc,
        "test_loss": test_loss,
        "test_accuracy": test_acc
    })

    print(
        f"Epoch [{epoch+1}/{epochs}] | "
        f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.2f}% | "
        f"Test Acc: {test_acc:.2f}%"
    )


Epoch [1/30] | Train Loss: 1.4550, Train Acc: 46.82% | Test Acc: 59.36%
Epoch [2/30] | Train Loss: 1.0399, Train Acc: 62.88% | Test Acc: 66.38%
Epoch [3/30] | Train Loss: 0.8794, Train Acc: 68.69% | Test Acc: 71.80%
Epoch [4/30] | Train Loss: 0.7898, Train Acc: 71.91% | Test Acc: 73.28%
Epoch [5/30] | Train Loss: 0.7416, Train Acc: 73.70% | Test Acc: 74.59%
Epoch [6/30] | Train Loss: 0.6967, Train Acc: 75.43% | Test Acc: 76.13%
Epoch [7/30] | Train Loss: 0.6549, Train Acc: 76.99% | Test Acc: 76.96%
Epoch [8/30] | Train Loss: 0.6265, Train Acc: 77.99% | Test Acc: 75.24%
Epoch [9/30] | Train Loss: 0.6069, Train Acc: 78.58% | Test Acc: 79.83%
Epoch [10/30] | Train Loss: 0.5818, Train Acc: 79.62% | Test Acc: 77.80%
Epoch [11/30] | Train Loss: 0.5608, Train Acc: 80.30% | Test Acc: 78.49%
Epoch [12/30] | Train Loss: 0.5440, Train Acc: 80.83% | Test Acc: 80.55%
Epoch [13/30] | Train Loss: 0.5310, Train Acc: 81.49% | Test Acc: 81.25%
Epoch [14/30] | Train Loss: 0.5155, Train Acc: 82.05% | Test